In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import pandas as pd 
import torch 
import torch.nn as nn 
import numpy as np
import wandb
from torch.utils.data import Dataset , DataLoader
import os 

In [3]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

import wandb
wandb.login()  

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23f2001025 (23f2001025-indian-institue-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [5]:
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [6]:
test.iloc[0]

id                                                        1
prompt    Pick the best possible answer: What is the rel...
A         For every eigenstate of one Hamiltonian, its p...
B         For every eigenstate of one Hamiltonian, its p...
C         For every eigenstate of one Hamiltonian, its p...
D         For every eigenstate of one Hamiltonian, its p...
E         For every eigenstate of one Hamiltonian, its p...
Name: 0, dtype: object

In [7]:
train

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [8]:
test

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [9]:
sam = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [10]:
sam

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C
...,...,...
495,496,A B C
496,497,A B C
497,498,A B C
498,499,A B C


In [11]:
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [12]:
LABEL_2_IDX = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX_2_LABEL = {v: k for k, v in LABEL_2_IDX.items()}
OPTIONS = ['A', 'B', 'C', 'D', 'E']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
train_df = train.copy()

In [14]:
train_df['answer'] = train_df['answer'].apply(lambda x:LABEL_2_IDX[x])

In [15]:
train_df

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,0
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,1
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,4
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,3
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,1


In [16]:
train_set,val_set = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

In [17]:
train_set = train_set.reset_index(drop=True)
val_set = val_set.reset_index(drop=True)

In [18]:
val_set

,id,prompt,A,B,C,D,E,answer
0,754,Select the most accurate option: What is the r...,"The Wigner function W(x, p) is the Wigner tran...","The Wigner function W(x, p) is a source functi...","The Wigner function W(x, p) is the derivative ...","The Wigner function W(x, p) represents the Ham...","The Wigner function W(x, p) is the time deriva...",0
1,11,Select the most accurate option: What is the P...,The Peierls bracket is a mathematical symbol u...,The Peierls bracket is a mathematical tool use...,The Peierls bracket is a Poisson bracket deriv...,The Peierls bracket is a mathematical symbol u...,The Peierls bracket is a mathematical tool use...,2
2,1364,Determine the correct option: What is the sign...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,2
3,1672,What is radiosity in radiometry?,Radiosity is the radiant flux entering a surfa...,Radiosity is the radiant flux entering a surfa...,Radiosity is the radiant flux leaving a surfac...,Radiosity is the radiant flux leaving a surfac...,Radiosity is the radiant flux leaving a surfac...,3
4,274,Pick the best possible answer: What is the rea...,The formation of stars occurs exclusively outs...,The low temperatures and high densities of mol...,The low temperatures and low densities of mole...,The high temperatures and low densities of mol...,The high temperatures and high densities of mo...,1
...,...,...,...,...,...,...,...,...
195,1017,Which of the following is correct? What is a p...,A mechanism of planets that are all located in...,A structure of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A structure of planets that are all located in...,A framework of planets that are all made of gas.,2
196,112,Pick the best possible answer: What is a Schwa...,A black hole that has mass but neither electri...,"A black hole that has mass, electric charge, a...",A black hole that has mass but neither electri...,A black hole that has neither mass nor electri...,"A black hole that has mass, electric charge, a...",2
197,1928,Choose the correct answer: What is magnetic su...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,1
198,428,Which of the following is correct? What is pow...,Power density is a measure of the amount of po...,Power density is a measure of the amount of po...,Power density is a measure of the amount of po...,Power density is a measure of the amount of po...,Power density is a measure of the amount of po...,1


In [19]:
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN = 256  

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded:', MODEL_NAME)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer loaded: microsoft/deberta-v3-base


In [20]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        choices = [str(row[opt]) for opt in OPTIONS]

        
        encodings = self.tokenizer(
            [prompt] * 5,
            choices,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        item = {
            'input_ids':      encodings['input_ids'],       
            'attention_mask': encodings['attention_mask'], 
        }
        if 'token_type_ids' in encodings:
            item['token_type_ids'] = encodings['token_type_ids']

        if not self.is_test:
            item['labels'] = torch.tensor(row['answer'], dtype=torch.long)

        return item

In [21]:
train_data = MCQDataset(train_set,tokenizer)
val_data = MCQDataset(val_set,tokenizer)


In [22]:
wt_path = "/kaggle/input/datasets/alokmanawat/loraweights/lora_weights_only.pth"

In [23]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                            # LoRA rank — increase for more capacity
    lora_alpha=32,                   # Scaling factor
    lora_dropout=0.1,
    target_modules=['query_proj', 'key_proj', 'value_proj'],  # DeBERTa attention layers
    bias='none'
)

"""base_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True
)



model = get_peft_model(base_model, lora_config)"""


# Load After training 

base_model = AutoModelForMultipleChoice.from_pretrained('microsoft/deberta-v3-base')
model = get_peft_model(base_model, lora_config)

model.load_state_dict(
    torch.load(wt_path),
    strict=False
)
model = model.to(device)




pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight              

In [24]:
def map_at_3(predictions, labels):
    map_score = 0.0
    for pred_top3, true_label in zip(predictions, labels):
        score = 0.0
        num_hits = 0
        for k, p in enumerate(pred_top3[:3], 1):
            if p == true_label:
                num_hits += 1
                score += num_hits / k
        map_score += score
    return map_score / len(labels)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # logits shape: (batch, 5)
    top3_preds = np.argsort(logits, axis=-1)[:, ::-1][:, :3]  # top 3 indices
    
    # Accuracy (top-1)
    top1_acc = (top3_preds[:, 0] == labels).mean()
    
    # MAP@3
    map3 = map_at_3(top3_preds, labels)
    
    return {'accuracy': top1_acc, 'map@3': map3}

In [25]:
""""training_args = TrainingArguments(
    output_dir='./deberta-mcq-lora',
    num_train_epochs=5,
    per_device_train_batch_size=4,      
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,    
    warmup_ratio=0.1,
    learning_rate=2e-4,                
    weight_decay=0.01,
    fp16=False,
    bf16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='map@3',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',                  
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting training...')
trainer.train()"""

'"training_args = TrainingArguments(\n    output_dir=\'./deberta-mcq-lora\',\n    num_train_epochs=5,\n    per_device_train_batch_size=4,      \n    per_device_eval_batch_size=8,\n    gradient_accumulation_steps=4,    \n    warmup_ratio=0.1,\n    learning_rate=2e-4,                \n    weight_decay=0.01,\n    fp16=False,\n    bf16=False,\n    eval_strategy=\'epoch\',\n    save_strategy=\'epoch\',\n    load_best_model_at_end=True,\n    metric_for_best_model=\'map@3\',\n    greater_is_better=True,\n    logging_steps=50,\n    report_to=\'none\',                  \n    seed=42\n)\n\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_data,\n    eval_dataset=val_data,\n    processing_class=tokenizer,\n    compute_metrics=compute_metrics,\n    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]\n)\n\nprint(\'Starting training...\')\ntrainer.train()'

In [26]:
""""torch.save(
    {k: v for k, v in model.state_dict().items() if 'lora' in k},
    '/kaggle/working/lora_weights_only.pth'
)"""

'"torch.save(\n    {k: v for k, v in model.state_dict().items() if \'lora\' in k},\n    \'/kaggle/working/lora_weights_only.pth\'\n)'

In [27]:
# Evaluate on validation set
#results = trainer.evaluate()
#print('\nValidation Results:')
#for k, v in results.items():
#    print(f'  {k}: {v:.4f}')

In [28]:
def predict_top3(model, tokenizer, df, max_len=256, batch_size=8):
    """Run inference and return top-3 predicted labels for each row."""
    model.eval()
    dataset = MCQDataset(df, tokenizer, max_len=max_len, is_test=True)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_logits = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

            outputs = model(**kwargs)
            all_logits.append(outputs.logits.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)  # (N, 5)

    # Rank options by logit score (descending)
    top3_indices = np.argsort(all_logits, axis=-1)[:, ::-1][:, :3]
    top3_labels  = [[IDX_2_LABEL[i] for i in row] for row in top3_indices]
    predictions  = [' '.join(labels) for labels in top3_labels]

    return predictions


print('Running inference on test set...')
test_preds = predict_top3(model, tokenizer, test, max_len=MAX_LEN)

Running inference on test set...


In [29]:
test

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [30]:
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': test_preds
})

submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(submission.head(10).to_string())

submission.csv saved!
   ID Prediction
0   1      A D C
1   2      B C D
2   3      E B D
3   4      E C A
4   5      E D A
5   6      D B A
6   7      C E D
7   8      A D B
8   9      D C E
9  10      E D B
